In [0]:
%pip install lightgbm xgboost
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "olist_dev")
CATALOG = dbutils.widgets.get("catalog")
TABLE_FEATURES = f"{CATALOG}.ml.features_commandes"
NOM_MODELE = f"{CATALOG}.ml.delivery_delay_model"
UTILISATEUR = spark.sql("SELECT current_user()").first()[0]
EXPERIENCE = f"/Users/{UTILISATEUR}/olist_retard_livraison"
print(TABLE_FEATURES, NOM_MODELE, EXPERIENCE, sep="\n")

In [0]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import mlflow
import pandas as pd
from mlflow import MlflowClient
from olist.train import (CIBLE, COLONNES_X, charger_jeux, choisir_seuil, construire_pipeline,
                         metriques, metriques_au_seuil, modeles_candidats)

mlflow.autolog(disable=True)
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIENCE)

In [0]:
jeux = charger_jeux(spark, TABLE_FEATURES)
X_tr, y_tr = jeux["entrainement"][COLONNES_X], jeux["entrainement"][CIBLE]
X_val, y_val = jeux["validation"][COLONNES_X], jeux["validation"][CIBLE]
X_te, y_te = jeux["test"][COLONNES_X], jeux["test"][CIBLE]

for nom, y in [("entrainement", y_tr), ("validation", y_val), ("test", y_te)]:
    print(f"{nom:<13} {len(y):>6,} commandes   {y.mean():.2%} en retard")
assert len(X_tr) == 64_020 and X_tr.shape[1] == 34

In [0]:
resultats = []
for nom, modele in modeles_candidats().items():
    with mlflow.start_run(run_name=nom) as run:
        pipe = construire_pipeline(nom, modele)
        pipe.fit(X_tr, y_tr)
        m = metriques(y_val, pipe.predict_proba(X_val)[:, 1])

        params = {k: v for k, v in modele.get_params().items()
                  if isinstance(v, (int, float, str, bool))}
        mlflow.log_params({"modele": nom, "n_features": len(COLONNES_X), **params})
        mlflow.log_metrics({f"val_{k}": v for k, v in m.items()})
        mlflow.sklearn.log_model(pipe, name="model", input_example=X_tr.head(5))

        resultats.append({"modele": nom, "run_id": run.info.run_id, **m})
        print(f"{nom:<14} PR-AUC val {m['pr_auc']:.4f}   hasard {m['prevalence']:.4f}"
              f"   ×{m['lift_pr_auc']:.1f}")

In [0]:
comparaison = (pd.DataFrame(resultats)
               .sort_values("pr_auc", ascending=False)
               .reset_index(drop=True))
display(comparaison.drop(columns="run_id"))
meilleur = comparaison.iloc[0]
print(f"meilleur sur la validation : {meilleur.modele}")

In [0]:
pipe = mlflow.sklearn.load_model(f"runs:/{meilleur.run_id}/model")

seuil = choisir_seuil(y_val, pipe.predict_proba(X_val)[:, 1])
proba_te = pipe.predict_proba(X_te)[:, 1]
m_te = metriques(y_te, proba_te)
s_te = metriques_au_seuil(y_te, proba_te, seuil["seuil"])

with mlflow.start_run(run_id=meilleur.run_id):
    mlflow.log_metrics({"seuil": seuil["seuil"],
                        **{f"val_seuil_{k}": v for k, v in seuil.items() if k != "seuil"},
                        **{f"test_{k}": v for k, v in m_te.items()},
                        **{f"test_seuil_{k}": v for k, v in s_te.items()}})

print(f"seuil {seuil['seuil']:.3f}  (validation : précision {seuil['precision']:.1%}, rappel {seuil['rappel']:.1%})")
print(f"TEST  PR-AUC {m_te['pr_auc']:.4f}  hasard {m_te['prevalence']:.4f}  ×{m_te['lift_pr_auc']:.1f}")
print(f"TEST  au seuil : {s_te['alertes']} alertes, {s_te['retards_detectes']} retards détectés, "
      f"{s_te['retards_rates']} ratés  (précision {s_te['precision']:.1%}, rappel {s_te['rappel']:.1%})")

In [0]:
version = mlflow.register_model(f"runs:/{meilleur.run_id}/model", NOM_MODELE)
client = MlflowClient()
client.set_registered_model_alias(NOM_MODELE, "champion", version.version)
client.set_model_version_tag(NOM_MODELE, version.version, "seuil", f"{seuil['seuil']:.6f}")
client.set_model_version_tag(NOM_MODELE, version.version, "algorithme", meilleur.modele)
print(f"{NOM_MODELE} v{version.version} → alias 'champion'")